# Q2 Hock-Brooke (Automobiles): warehouse size by simulation

New cars arrive on **Monday** and are collected on Friday or Saturday. So during the week the dealer must store
the cars left over from last week **plus** this week's delivery. Warehouse spaces cost £10 per week each. Any car
beyond the warehouse capacity needs a hired parking space at £20 per week.

| Cars delivered | 7 | 8 | 9 | 10 |
| --- | --- | --- | --- | --- |
| Frequency (%) | 10 | 20 | 50 | 20 |

| Weekly demand | 6 | 7 | 8 | 9 | 10 | 11 |
| --- | --- | --- | --- | --- | --- | --- |
| Frequency (%) | 5 | 20 | 30 | 10 | 20 | 15 |

*Made for this site (erratum C1). The workbook `Questions 2 and 5 - forecasting and simulation.xlsx` originally had the two
tables swapped, and the demand values shifted by one, so its "long-run" costs answered a different question. This notebook
uses the data as printed in `Revision.pdf`. It charges overflow on the start-of-week stock (opening stock + Monday delivery),
and it starts with an empty warehouse.*

In [1]:
import random
import numpy as np

**The model for one week.** Draw the delivery and the demand from the tables. Stock this week = leftover + delivery. Cost = 10 × capacity + 20 × max(0, stock − capacity). Sales = min(stock, demand), and unsold cars carry over.

In [2]:
DELIVERED = ([7, 8, 9, 10], [0.10, 0.20, 0.50, 0.20])
DEMAND = ([6, 7, 8, 9, 10, 11], [0.05, 0.20, 0.30, 0.10, 0.20, 0.15])
SPACE_COST, PARKING_COST = 10, 20

def simulate(capacity, weeks=52, seed=1, show=0):
    """Average weekly cost of one run; prints the first `show` weeks as a table."""
    rng = random.Random(seed)
    leftover, total = 0, 0
    if show:
        print(f"{'Week':>4} {'Opening':>8} {'Delivered':>9} {'Stock':>6} {'Demand':>7} {'Sales':>6} {'Closing':>8} {'Overflow':>9} {'Cost':>5}")
    for week in range(1, weeks + 1):
        delivered = rng.choices(*DELIVERED)[0]
        demand = rng.choices(*DEMAND)[0]
        stock = leftover + delivered
        overflow = max(0, stock - capacity)
        cost = SPACE_COST * capacity + PARKING_COST * overflow
        sales = min(stock, demand)
        if week <= show:
            print(f"{week:>4} {leftover:>8} {delivered:>9} {stock:>6} {demand:>7} {sales:>6} {stock - sales:>8} {overflow:>9} {cost:>5}")
        leftover = stock - sales
        total += cost
    return total / weeks

mean_delivered = np.dot(*DELIVERED); mean_demand = np.dot(*DEMAND)
print(f"Mean delivered {mean_delivered:.2f} cars/week, mean demand {mean_demand:.2f}: stock grows by about "
      f"{mean_delivered - mean_demand:.2f} cars/week on average")

Mean delivered 8.80 cars/week, mean demand 8.65: stock grows by about 0.15 cars/week on average


## (a) Capacity 9: one illustrative year

The first 12 weeks of one 52-week run, then the average weekly cost for that run.

In [3]:
avg = simulate(9, weeks=52, seed=1, show=12)
print(f"\nAverage weekly cost over 52 weeks (this run): £{avg:.2f}")

Week  Opening Delivered  Stock  Demand  Sales  Closing  Overflow  Cost
   1        0         8      8      10      8        0         0    90
   2        0         9      9       8      8        1         0    90
   3        1         9     10       8      8        2         1   110
   4        2         9     11      10     10        1         2   130
   5        1         7      8       6      6        2         0    90
   6        2        10     12       8      8        4         3   150
   7        4         9     13       6      6        7         4   170
   8        7         9     16      10     10        6         7   230
   9        6         8     14      11     11        3         5   190
  10        3        10     13       6      6        7         4   170
  11        7         7     14       8      8        6         5   190
  12        6        10     16       8      8        8         7   230

Average weekly cost over 52 weeks (this run): £351.54


## (b) Is 9 the best capacity?

One run is noisy, so each capacity is simulated 2,000 times (52 weeks each) and the costs are averaged.
Deliveries exceed demand on average, so the leftover stock grows during the year. The answer therefore depends on the
horizon: the 12-week column shows what a one-quarter view would choose.

In [4]:
reps = 2000
print(f"{'Capacity':>8} {'52 weeks (£/week)':>18} {'12 weeks (£/week)':>18}")
res = {}
for cap in range(6, 19):
    c52 = np.mean([simulate(cap, 52, seed=s) for s in range(reps)])
    c12 = np.mean([simulate(cap, 12, seed=s) for s in range(reps)])
    res[cap] = (c52, c12)
    print(f"{cap:>8} {c52:>18.2f} {c12:>18.2f}")
best52 = min(res, key=lambda c: res[c][0]); best12 = min(res, key=lambda c: res[c][1])
print(f"\nBest over 52 weeks: {best52} spaces; best over 12 weeks: {best12} spaces")

Capacity  52 weeks (£/week)  12 weeks (£/week)


       6             271.28             168.65


       7             261.28             158.65


       8             251.56             149.33


       9             242.51             141.57


...

      14             220.94             149.30


      15             220.54             156.24


      16             221.25             164.12


      17             222.95             172.64


      18             225.61             181.66

Best over 52 weeks: 15 spaces; best over 12 weeks: 11 spaces


**Answer.** Over a year, the expected cost is lowest at about **15 spaces** (≈ £221.6 per week). 14 and 16 are within
£1, while 9 spaces cost ≈ £243.8 per week. Exact values, from a Markov-chain calculation: 9 → 243.78, 14 → 222.01,
15 → 221.56, 16 → 222.22. Over only 12 weeks the best is 11 spaces (≈ £137.9). In practice the dealer should also
order fewer cars, because stock that keeps growing is the real problem.